<a href="https://colab.research.google.com/github/Noors-lab/Toy-DETR/blob/main/toy_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports**


In [ ]:
import torch
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



Function to create one *image*

In [ ]:
def generate_image(image_size=64, max_objects=3):
    image = np.zeros((image_size, image_size, 3), dtype=np.uint8)

    num_objects = random.randint(1, max_objects)

    boxes = []
    labels = []

    for _ in range(num_objects):
        shape_type = random.randint(0, 1)  # 0=square, 1=circle

        x = random.randint(5, image_size - 20)
        y = random.randint(5, image_size - 20)
        w = random.randint(10, 20)
        h = w

        if shape_type == 0:
            cv2.rectangle(image, (x, y), (x+w, y+h), (255, 255, 255), -1)
        else:
            radius = w // 2
            cv2.circle(image, (x + radius, y + radius), radius, (255, 255, 255), -1)

        boxes.append([x, y, w, h])
        labels.append(shape_type)

    return image, boxes, labels



static dataset

In [ ]:
samples = []

for _ in range(20):  # small dataset
    image, boxes, labels = generate_image()

    samples.append({
        "image": image,
        "boxes": boxes,
        "labels": labels
    })


dataset class

In [ ]:
class StaticShapeDataset(Dataset):
    def __init__(self, samples):
        self.data = samples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        image = sample["image"]
        boxes = sample["boxes"]
        labels = sample["labels"]

        # convert image
        image = torch.tensor(image, dtype=torch.float32) / 255.0
        image = image.permute(2, 0, 1)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

        return image, target

collate_fn


In [ ]:
def collate_fn(batch):
    images = []
    targets = []

    for img, tgt in batch:
        images.append(img)
        targets.append(tgt)

    images = torch.stack(images)

    return images, targets

dataloader

In [ ]:
dataset = StaticShapeDataset(samples)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

testing

In [ ]:
for images, targets in dataloader:
    print("Images shape:", images.shape)
    print("Targets:", targets)
    break

Images shape: torch.Size([4, 3, 64, 64])
Targets: [{'boxes': tensor([[39., 32., 15., 15.],
        [38., 11., 12., 12.],
        [21., 11., 18., 18.]]), 'labels': tensor([1, 1, 1])}, {'boxes': tensor([[12., 10., 17., 17.]]), 'labels': tensor([0])}, {'boxes': tensor([[23., 42., 15., 15.],
        [12., 28., 19., 19.],
        [12.,  5., 17., 17.]]), 'labels': tensor([0, 1, 1])}, {'boxes': tensor([[43., 27., 19., 19.]]), 'labels': tensor([1])}]


**SINCE I DIDNT USE REAL IMAGE SO UP UNTILL THIS PART ISNT OUR MAIN CONCERN ... THE BUILDING FOR DETR STARTS FROM HERE**

SIMPLE CNN BACKBONE

In [ ]:
class SimpleCNNBackbone(nn.Module):
  def __init__(self):
      super().__init__()

      self.conv = nn.Sequential(nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

  def forward(self,x):
    return self.conv(x)

testing

In [ ]:
backbone = SimpleCNNBackbone()

for images, targets in dataloader:
    features = backbone(images)
    print(features.shape)
    break

torch.Size([4, 128, 16, 16])


Flatten

In [ ]:
def flatten(x):
  B,C,H,W = x.shape

  x = x.view(B,C,H*W)
  x = x.permute(0,2,1)
  return x

testing

In [ ]:
for images, _ in dataloader:
    features = backbone(images)
    tokens = flatten(features)

    print(tokens.shape)
    break

torch.Size([4, 256, 128])


positional encoding

In [ ]:
class positional_encoding(nn.Module):
  def __init__(self,dim,max_len=256):
    super().__init__()
    self.pos = nn.Parameter(torch.randn(1,max_len,dim))

  def forward(self,x):
    return x+self.pos[:, : x.size(1),:]


In [ ]:
pos_enc = positional_encoding(dim=128)
tokens = pos_enc(tokens)
print(tokens.shape)

torch.Size([4, 256, 128])


transformer encoder

In [ ]:
class Transformer_encoder(nn.Module):
  def __init__(self,dim=128,num_heads=4,num_layers=2):
    super().__init__()

    encoder_layer = nn.TransformerEncoderLayer(d_model=dim,
                                               nhead=num_heads,
                                               batch_first=True
                                               )
    self.encoder =nn.TransformerEncoder(encoder_layer,num_layers=num_layers)

  def forward(self,x):
      return self.encoder(x)


testing

In [ ]:
encoder = Transformer_encoder()

for images, _ in dataloader:
    features = backbone(images)
    tokens = flatten(features)
    tokens = pos_enc(tokens)

    encoded = encoder(tokens)

    print(encoded.shape)
    break

torch.Size([4, 256, 128])


defining queries

In [ ]:
num_queries = 10
embed_queries = nn.Parameter(torch.randn(1,num_queries,128))

Transformer decoder

In [ ]:
class Transformer_decoder(nn.Module):
  def __init__(self,dim=128,num_heads=4,num_layers=2):
    super().__init__()

    decoder_layer = nn.TransformerDecoderLayer(d_model=dim,nhead=num_heads,batch_first=True)

    self.decoder = nn.TransformerDecoder(decoder_layer,num_layers=num_layers)


  def forward(self,queries,memory):
    return self.decoder(queries,memory)

testing

In [ ]:
from IPython.utils.py3compat import encode
decoder = Transformer_decoder()

for images, _ in dataloader:
    features = backbone(images)
    tokens = flatten(features)
    tokens = pos_enc(tokens)

    memory = encoder(tokens)

    queries = embed_queries.expand(images.size(0), -1, -1)

    output = decoder(queries, memory)

    print(output.shape)
    break

torch.Size([4, 10, 128])


prediction heads

In [ ]:
class DETRHead(nn.Module):
    def __init__(self, dim=128, num_classes=2):
        super().__init__()

        self.class_head = nn.Linear(dim, num_classes + 1)
        # +1 for "no object"

        self.box_head = nn.Linear(dim, 4)  # x, y, w, h

    def forward(self, x):
        class_logits = self.class_head(x)
        boxes = self.box_head(x)

        return class_logits, boxes

full forward pass

In [ ]:
backbone = SimpleCNNBackbone()
encoder = Transformer_encoder()
decoder = Transformer_decoder()
head = DETRHead()

for images, _ in dataloader:
    features = backbone(images)
    tokens = flatten(features)
    tokens = pos_enc(tokens)

    memory = encoder(tokens)

    queries = embed_queries.expand(images.size(0), -1, -1)

    hs = decoder(queries, memory)

    class_logits, boxes = head(hs)

    print(class_logits.shape, boxes.shape)
    break

torch.Size([4, 10, 3]) torch.Size([4, 10, 4])
